In [ ]:
from morphology_pipeline import pipeline
from tifffile import imread

background = r'..\test_1462.tif'
img = imread(background)
img = img[..., :3]
folder = r'..\Chip_1462_1_real'

SD, mask, selected_objs, window_desc = pipeline.run_pipeline_and_save_csvs(
    img,
    None,
    None,
    folder
)



c:\Users\sbsas\Documents\uni\Projects\Nikita PhD\morphology_pipeline\.venv\lib\site-packages\cp_measure\core\measureobjectsizeshape.py:637: RuntimeWarning: divide by zero encountered in divide
  formfactor = 4.0 * numpy.pi * props["area"] / props["perimeter"] ** 2


Detected rotation: 28.341 degrees
Detected 23 corridors.
Total matched windows: 609


In [5]:
import json
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

res = SD.dataframe.loc[mask]
res.to_csv("res/cell_data.csv")

res.info()

with open("res/window_desc.json", "w") as f:
    json.dump(window_desc, f, cls=NumpyEncoder)


<class 'pandas.core.frame.DataFrame'>
Index: 207 entries, 9 to 727
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   path_background  207 non-null    object
 1   path_dapi        207 non-null    object
 2   path_yap         207 non-null    object
 3   path_actin       207 non-null    object
 4   height           207 non-null    int64 
 5   width            207 non-null    int64 
 6   channels         207 non-null    object
 7   center           207 non-null    object
 8   eccentricity     207 non-null    object
 9   center_rot       207 non-null    object
dtypes: int64(2), object(8)
memory usage: 17.8+ KB


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ast import literal_eval

def overlay_cells_dataframe(df, background_path, alpha=0.4, figsize=(8,8)):
    """
    Overlay all cell channels in a dataframe onto a single background image.
    Plots both center and center_rot points if present.

    Parameters
    ----------
    df : pandas.DataFrame
        Must include paths and 'center'/'center_rot' columns.
    background_path : str
        Path to background image.
    alpha : float
        Transparency for overlays.
    """
    
    # --- Load background ---
    bg = cv2.imread(background_path, cv2.IMREAD_GRAYSCALE)
    if bg is None:
        raise ValueError("Background image could not be loaded.")
    
    bg = cv2.cvtColor(bg, cv2.COLOR_GRAY2RGB).astype(float)

    # Canvas to accumulate overlays
    combined = bg.copy()

    # --- Process each cell row ---
    for idx, row in df.iterrows():
        for ch in ["path_dapi", "path_yap", "path_actin"]:
            path = row[ch]
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                continue

            # Convert to RGB (false color)
            colored = cv2.applyColorMap(img, cv2.COLORMAP_JET).astype(float)
            
            # Blend
            combined = cv2.addWeighted(combined, 1, colored, alpha, 0)

    # Clip to valid range
    combined = np.clip(combined, 0, 255).astype(np.uint8)

    # --- Visualization ---
    plt.figure(figsize=figsize)
    plt.imshow(combined)
    plt.axis("off")

    # --- Plot all centers ---
    for idx, row in df.iterrows():
        # SAFELY parse tuple stored as string
        try:
            cx, cy = literal_eval(row["center"])
            plt.scatter(cx, cy, c="white", s=20, edgecolors="black")
        except:
            pass
        
        # Optional rotated center
        try:
            rx, ry = literal_eval(row["center_rot"])
            plt.scatter(rx, ry, c="yellow", s=20, edgecolors="black")
        except:
            pass

    plt.show()

overlay_cells_dataframe(res, background)

In [ ]:
df_identifyPrimaryObjects = pd.read_csv("datasets/MyExpt_IdentifyPrimaryObjects.csv")
df_Phalloidin_segmented = pd.read_csv("datasets/MyExpt_Phalloidin_segmented.csv")
df_YAP_segmented = pd.read_csv("datasets/MyExpt_YAP_segmented.csv")

SD_clean["ImageNumber"] = SD_clean.index // 10000
SD_clean["ObjectNumber"] = SD_clean.index % 10000

keys = ["Metadata_ObjNumber", "ObjectNumber"]

cluster_map = SD_clean[
    ["ImageNumber", "ObjectNumber", "cluster_hdbscan", "pseudotime", "pseudotime_widths", "pseudotime_center_bin"]
].rename(columns={"ImageNumber": "Metadata_ObjNumber"})

def prepare_df(df, prefix, keys):
    # rename all non-key columns with a prefix
    rename_dict = {
        col: f"{prefix}_{col}"
        for col in df.columns
        if col not in keys
    }
    return df.rename(columns=rename_dict)

# prefix each dataframe to avoid collisions
df_id = prepare_df(df_identifyPrimaryObjects, "dapi", keys)
df_ph = prepare_df(df_Phalloidin_segmented, "phalloidin", keys)
df_yap = prepare_df(df_YAP_segmented, "yap", keys)

# merge everything into one dataframe
df_all = (
    cluster_map
    .merge(df_id, on=keys, how="left")
    .merge(df_ph, on=keys, how="left")
    .merge(df_yap, on=keys, how="left")
)

print(df_all.columns)
print(df_all.head())

df_all.to_csv("datasets/combined_clustered_data 1476.csv", index=False)